# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MTahaR-dev/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** This notebook develops a small March 2026 decline-risk ranking prototype. It uses real warehouse data, exactly three verification queries, five features, and one deliberately leaked column that is removed before the final score.

I used AI assistance with the repo's `writing-data-contracts` and `flyrank/flyrank-data` skills. The notebook keeps the distinction between observed decline risk and the unproven benefit of refreshing a page.

**Access:** Accept the gate at https://huggingface.co/datasets/FlyRank/internship-warehouse, then provide a READ token through the `HF_TOKEN` environment variable, a Colab Secret named `HF_TOKEN`, or an existing local Hugging Face login. Never put a token in notebook source or output. A previously downloaded March file can also be used without a token.

Run all cells in order. The only warehouse partition downloaded is `month=2026-03`; June's final-month sample is not read. Raw downloads remain in a local data/cache folder and must not be committed.

## 1. Unit of analysis + time window

My contract has five plain-word answers:

1. **What one row means:** A source row is one report date × pseudonymized client × pseudonymized content item. After aggregation, one modeling row is one client/content pair at the fixed decision date **2026-03-18**. IDs are grouping context, not features.
2. **Which table:** Only the March partition of `fact_content_daily_performance` in the fixed warehouse release. This prototype needs search measurements and their availability flag, so no dimension or query-table join is necessary.
3. **Which window:** The partition is March 1–31, 2026. Features use March 1–14 (14 days). March 15–17 is a three-day reporting buffer. The decision is March 18; the outcome window is March 18–31 (14 days). The buffer follows the release's conservative freshness practice, but historical data-arrival timestamps are not supplied, so actual availability at each decision cannot be proven from this frozen snapshot.
4. **What I would predict/rank:** Rank pages by risk of an observed later-window impression decline. The binary proxy is 1 if outcome-window impressions are more than 20% below feature-window impressions, otherwise 0. Both windows are the same length. The reference-volume floor is 100 impressions in the feature window, a provisional policy choice. Complete-case retrospective evaluation additionally requires 14 valid GSC rows in each window; outcome coverage is label eligibility, never a model feature.
5. **One deliberate exclusion:** The fixed 90-day query table is excluded because its snapshot-end window does not align with March and could contain later information. Outcome counts, the target, and any label-derived column are also excluded from the final five inputs.

**Output and action:** A within-client review queue with a risk score and measured reasons. An editor investigates the page and context, then decides whether to update, investigate further, or monitor. A wrong recommendation can waste editorial time; the proxy does not estimate the benefit of editing.

In [1]:
# Install only missing libraries; no credentials appear in this command.
import importlib.util
import subprocess
import sys

packages = {"duckdb": "duckdb",
            "sklearn": "scikit-learn", "pandas": "pandas"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import os
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

MONTH = "2026-03"
REPO_ID = "FlyRank/internship-warehouse"
REVISION = "50cbf7c3909d07be4d1b5906b4d09e882e5acbf2"
FILENAME = f"fact_content_daily_performance/month={MONTH}/data_0.parquet"

# Find a repo root if present; otherwise use the current Colab folder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "docs/ml-intern-dataset-and-lane-guide.md").is_file()), Path.cwd())
local_candidates = [
    root / "data/warehouse" / f"month={MONTH}" / "data_0.parquet",
    root / "data/warehouse" / FILENAME,
]
month_file = next((p for p in local_candidates if p.is_file()), None)

if month_file is None:
    if importlib.util.find_spec("huggingface_hub") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import get_token, hf_hub_download
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    token = token or get_token()
    if not token:
        raise RuntimeError("Warehouse access is required: set HF_TOKEN securely, "
                           "use a local Hugging Face login, or supply the March parquet file. "
                           "Do not paste a token into this notebook.")
    month_file = Path(hf_hub_download(
        repo_id=REPO_ID, repo_type="dataset", filename=FILENAME,
        revision=REVISION, token=token, cache_dir=str(root / "data/warehouse_cache"),
    ))
    del token

con = duckdb.connect()
con.execute("SET threads = 2")
con.execute("SET memory_limit = '1GB'")
temp_folder = root / "data/warehouse_cache/duckdb_tmp"
temp_folder.mkdir(parents=True, exist_ok=True)
con.execute("SET temp_directory = ?", [str(temp_folder)])
# This is a local cached file, so repeat verification does not rescan Hugging Face.
con.read_parquet(str(month_file)).create_view("march_raw")
required_columns = {
    "report_date", "client_hash_id", "content_hash_id", "gsc_data_available",
    "ga4_data_available", "gsc_impressions", "gsc_clicks", "gsc_avg_position",
}
observed_columns = set(con.table("march_raw").columns)
assert required_columns <= observed_columns, "Warehouse schema differs; inspect before proceeding."
print("Loaded the cached March partition; required field names verified.")

Loaded the cached March partition; required field names verified.


## 2. Fields: feature / label / context / excluded

Every field used has one role:

| Role | Fields | Treatment |
|---|---|---|
| Feature sources | March 1–14 `gsc_impressions`, `gsc_clicks`, valid `gsc_avg_position` | Aggregate into exactly five numeric inputs listed below. |
| Label/proxy | March 18–31 `gsc_impressions`; derived `future_decline_proxy` | Outcome evidence and target only; never model inputs. |
| Context | `report_date`, `client_hash_id`, `content_hash_id`, decision date, GSC coverage counts | Windows, grouping, eligibility, client holdout, and explanations. |
| Context | `gsc_data_available`, `ga4_data_available` | Availability auditing. GSC must be `IS TRUE`; GA4 is reported but is not required because no GA4 feature is used. |
| Excluded | March 15–17 measurements, all non-feature dates in `X`, query-table metrics, product flags, raw text/URLs | Reporting buffer, future leakage, incompatible windows, circular answers, or privacy. |
| Excluded after demonstration | `leak_target_copy` | Exactly copies the target and is deliberately invalid. |

**Missingness policy:** `FALSE` or `NULL` availability never means measured zero traffic. A day is usable only when `gsc_data_available IS TRUE` and its impressions/clicks are non-null and non-negative. I require 14 such days per window for this retrospective labeled slice. Position contributes only when impressions are positive and position is non-null and greater than zero. If no valid position remains, the aggregated feature is missing and is median-imputed using training clients only. Missingness and coverage are shown in the verification outputs; I do not fill unavailable daily traffic with zeros.

## 3. Three verification queries, five features, and the leakage trap

The next three cells are the **three verification queries**. They check the raw grain, March's size/date span, and availability/missingness. A later aggregation query builds the feature frame; it is not a fourth verification query.

### Verification query 1 — source grain

Zero returned rows means no duplicate report-date/client/content keys were found. Query 2 separately checks for null keys.

In [2]:
grain_query = """
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS duplicate_count
FROM march_raw
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""
grain_check = con.execute(grain_query).df()
print(f"Duplicate grain keys returned: {len(grain_check)}")
assert grain_check.empty, "Resolve duplicates before aggregating."
grain_check

Duplicate grain keys returned: 0


,report_date,client_hash_id,content_hash_id,duplicate_count


### Verification query 2 — row count and date span

The row count comes from this partition, not the guide's full-release count. The date checks must confirm March only, with 31 distinct dates and no null grain keys.

In [3]:
count_query = """
SELECT COUNT(*) AS source_rows,
       MIN(CAST(report_date AS DATE)) AS min_date,
       MAX(CAST(report_date AS DATE)) AS max_date,
       COUNT(DISTINCT CAST(report_date AS DATE)) AS distinct_dates,
       COUNT(DISTINCT client_hash_id) AS clients,
       COUNT(DISTINCT content_hash_id) AS content_items,
       COUNT(*) FILTER (WHERE report_date IS NULL OR client_hash_id IS NULL
                         OR content_hash_id IS NULL) AS null_grain_keys
FROM march_raw
"""
count_check = con.execute(count_query).df()
assert count_check.loc[0, "source_rows"] > 0
assert pd.Timestamp(count_check.loc[0, "min_date"]) == pd.Timestamp("2026-03-01")
assert pd.Timestamp(count_check.loc[0, "max_date"]) == pd.Timestamp("2026-03-31")
assert count_check.loc[0, "distinct_dates"] == 31
assert count_check.loc[0, "null_grain_keys"] == 0
count_check

,source_rows,min_date,max_date,distinct_dates,clients,content_items,null_grain_keys
0,9841378,2026-03-01,2026-03-31,31,55,331437,0


### Verification query 3 — availability with `IS TRUE`

The all-client row shows how many rows survive the GSC availability filter and then the valid-count checks. The same query reports client-level differences and separates true, false, and unknown flags. GA4 counts are context only; this search-only prototype does not require GA4 tracking.

In [4]:
availability_query = """
SELECT CASE WHEN GROUPING(client_hash_id) = 1 THEN 'ALL_CLIENTS'
            ELSE client_hash_id END AS client_group,
       COUNT(*) AS source_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_true_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS gsc_false_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS gsc_unknown_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_true_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_false_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS ga4_unknown_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND ga4_data_available IS TRUE) AS both_true_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND (gsc_impressions IS NULL OR gsc_clicks IS NULL))
                         AS gsc_true_missing_counts,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND (gsc_impressions < 0 OR gsc_clicks < 0))
                         AS gsc_true_negative_counts,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND gsc_impressions IS NOT NULL AND gsc_clicks IS NOT NULL
                         AND gsc_impressions >= 0 AND gsc_clicks >= 0)
                         AS usable_gsc_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
                         AND (gsc_avg_position IS NULL OR gsc_avg_position <= 0))
                         AS positive_impression_rows_without_valid_position
FROM march_raw
GROUP BY GROUPING SETS ((), (client_hash_id))
ORDER BY CASE WHEN GROUPING(client_hash_id) = 1 THEN 0 ELSE 1 END, client_group
"""
availability_check = con.execute(availability_query).df()
overall = availability_check.loc[availability_check.client_group.eq("ALL_CLIENTS")].iloc[0]
assert overall.source_rows == count_check.loc[0, "source_rows"]
assert overall.gsc_true_rows + overall.gsc_false_rows + overall.gsc_unknown_rows == overall.source_rows
assert overall.ga4_true_rows + overall.ga4_false_rows + overall.ga4_unknown_rows == overall.source_rows
assert overall.usable_gsc_rows > 0
print(f"GSC IS TRUE: {int(overall.gsc_true_rows):,} of {int(overall.source_rows):,} rows "
      f"({overall.gsc_true_rows / overall.source_rows:.2%}).")
print(f"After valid-count checks: {int(overall.usable_gsc_rows):,} rows.")
print("Showing the overall row and the first nine pseudonymized client summaries.")
availability_check.head(10)

GSC IS TRUE: 3,611,061 of 9,841,378 rows (36.69%).
After valid-count checks: 3,611,061 rows.
Showing the overall row and the first nine pseudonymized client summaries.


,client_group,source_rows,gsc_true_rows,gsc_false_rows,gsc_unknown_rows,ga4_true_rows,ga4_false_rows,ga4_unknown_rows,both_true_rows,gsc_true_missing_counts,gsc_true_negative_counts,usable_gsc_rows,positive_impression_rows_without_valid_position
0,ALL_CLIENTS,9841378,3611061,6230317,0,413966,6408671,3018741,364347,0,0,3611061,163189
1,client_0797ff3a1fc9a6a5,8060,646,7414,0,0,0,8060,0,0,0,646,1
2,client_08a6a72ff48e62c0,851275,359419,491856,0,0,0,851275,0,0,0,359419,14891
3,client_08d2847f24cf89c1,12369,1374,10995,0,366,12003,0,124,0,0,1374,8
4,client_0e1acc6cd57b0eba,4557,209,4348,0,130,4427,0,13,0,0,209,1
5,client_0fa64a184f18a4a0,44197,21408,22789,0,3101,41096,0,2965,0,0,21408,1059
6,client_157ffe4d4a595515,133632,67439,66193,0,5892,95228,32512,3442,0,0,67439,3054
7,client_19b89ee4fe3db6da,213001,0,213001,0,3,212998,0,0,0,0,0,0
8,client_1a730cb2640a1abf,66583,38146,28437,0,7169,55893,3521,6186,0,0,38146,2602
9,client_20259bd6705d81d4,136720,119463,17257,0,29851,106869,0,28797,0,0,119463,2769


### Five features — available when?

All five features use only March 1–14. At the March 18 decision, that window has ended and three buffer days have elapsed. The release is retrospective, so this is a reporting-lag assumption, not proof of historical ingestion timestamps.

| Feature | Knowable at the decision moment because… |
|---|---|
| `log_impressions` | It is log(1 + impressions) from the completed pre-decision window only. |
| `log_clicks` | It is log(1 + clicks) from that same completed pre-decision window only. |
| `ctr_pct` | It is 100 × pre-decision clicks / pre-decision impressions, with a positive exposure floor. |
| `weighted_position` | It uses valid daily positions and their impression weights from the pre-decision window only. It is an aggregate approximation; exact within-day weighting is unavailable. |
| `prior_week_change_pct` | It compares impressions in March 8–14 with March 1–7; both seven-day subwindows ended before the decision date. A zero first-week denominator is treated as missing, not an infinite trend. |

The label compares equal-length later and earlier windows. No later-window total, coverage count, ID, or leaked column enters the five-feature frame. CTR uses percentage units.

In [5]:
# Feature-building aggregation, separate from the three verification queries.
feature_query = """
WITH usable AS (
    SELECT CAST(report_date AS DATE) AS day, client_hash_id, content_hash_id,
           gsc_impressions, gsc_clicks, gsc_avg_position
    FROM march_raw
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions IS NOT NULL AND gsc_clicks IS NOT NULL
      AND gsc_impressions >= 0 AND gsc_clicks >= 0
), windowed AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (WHERE day BETWEEN DATE '2026-03-01' AND DATE '2026-03-14') AS feature_days,
           COUNT(*) FILTER (WHERE day BETWEEN DATE '2026-03-18' AND DATE '2026-03-31') AS outcome_days,
           SUM(gsc_impressions) FILTER (WHERE day BETWEEN DATE '2026-03-01' AND DATE '2026-03-14') AS feature_impressions,
           SUM(gsc_clicks) FILTER (WHERE day BETWEEN DATE '2026-03-01' AND DATE '2026-03-14') AS feature_clicks,
           SUM(gsc_impressions) FILTER (WHERE day BETWEEN DATE '2026-03-18' AND DATE '2026-03-31') AS outcome_impressions,
           SUM(gsc_avg_position * gsc_impressions) FILTER (
               WHERE day BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
                 AND gsc_impressions > 0 AND gsc_avg_position > 0) AS position_numerator,
           SUM(gsc_impressions) FILTER (
               WHERE day BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
                 AND gsc_impressions > 0 AND gsc_avg_position > 0) AS position_denominator,
           SUM(gsc_impressions) FILTER (WHERE day BETWEEN DATE '2026-03-01' AND DATE '2026-03-07') AS first_week_impressions,
           SUM(gsc_impressions) FILTER (WHERE day BETWEEN DATE '2026-03-08' AND DATE '2026-03-14') AS second_week_impressions
    FROM usable
    GROUP BY client_hash_id, content_hash_id
)
SELECT *, DATE '2026-03-18' AS decision_date
FROM windowed
WHERE feature_days = 14 AND outcome_days = 14 AND feature_impressions >= 100
ORDER BY client_hash_id, content_hash_id
"""
window_frame = con.execute(feature_query).df()
assert not window_frame.empty, "No complete labeled pages; revisit coverage before modeling."
assert not window_frame.duplicated(["client_hash_id", "content_hash_id", "decision_date"]).any()

features = pd.DataFrame({
    "log_impressions": np.log1p(window_frame.feature_impressions),
    "log_clicks": np.log1p(window_frame.feature_clicks),
    "ctr_pct": 100 * window_frame.feature_clicks / window_frame.feature_impressions,
    "weighted_position": window_frame.position_numerator / window_frame.position_denominator,
    "prior_week_change_pct": 100 * (
        window_frame.second_week_impressions - window_frame.first_week_impressions
    ) / window_frame.first_week_impressions.replace(0, np.nan),
}).replace([np.inf, -np.inf], np.nan)
FEATURES = list(features.columns)
assert len(FEATURES) == 5
assert features.notna().any().all(), "A feature has no observed values; inspect the slice."
y = window_frame.outcome_impressions.lt(0.8 * window_frame.feature_impressions).astype("int64")
y.name = "future_decline_proxy"
groups = window_frame.client_hash_id
assert y.nunique() == 2, "Both target classes are needed for the quick experiment."

print(f"Complete labeled feature rows: {len(features):,}; clients: {groups.nunique()}.")
print(f"Observed decline proxy: {int(y.sum()):,} pages ({y.mean():.2%}).")
print("Missing values in the five features:")
print(features.isna().sum().to_string())
print("Distinct values per feature (to identify constant inputs):")
print(features.nunique().to_string())
print("Five-feature frame below; identifiers and outcome fields are stored separately.")
features.head(8)

Complete labeled feature rows: 57,459; clients: 32.
Observed decline proxy: 21,053 pages (36.64%).
Missing values in the five features:
log_impressions          0
log_clicks               0
ctr_pct                  0
weighted_position        0
prior_week_change_pct    0
Distinct values per feature (to identify constant inputs):
log_impressions           8318
log_clicks                 223
ctr_pct                  16224
weighted_position        55457
prior_week_change_pct    39897
Five-feature frame below; identifiers and outcome fields are stored separately.


,log_impressions,log_clicks,ctr_pct,weighted_position,prior_week_change_pct
0,4.744932,0.693147,0.877193,12.421053,19.230769
1,5.609472,0.000000,0.000000,12.466912,-2.898551
2,5.991465,1.098612,0.501253,11.706767,-13.551402
3,7.364547,2.708050,0.887199,7.308619,-63.376623
4,7.167809,0.000000,0.000000,56.003858,-15.647226
5,7.218177,1.386294,0.220103,8.975055,102.888889
6,4.934474,0.693147,0.724638,10.811594,33.898305
7,6.731018,0.000000,0.000000,9.485066,304.216867


### Quick honest score, then one deliberate leak

I split whole clients with a fixed random seed and keep the exact same train/test rows for all three runs. Median imputation is fitted on training data only. The quick classifier is a depth-3 decision tree; no tuning or model selection is performed on the test set.

The diagnostic score is **ROC-AUC** (0.5 is chance ordering; 1.0 is perfect ordering). My Lane 2 queue metric remains **mean per-client Precision@20**, also reported for test clients with at least 20 pages. The client split tests transfer across clients within one month; it is not yet a future-month evaluation.

The single deliberate leak is `leak_target_copy = y`: it directly exposes each page's later-window answer. It should drive the held-out score toward perfect even though the real forecasting problem remains unsolved. I then remove it, discard the leaked model, retrain on exactly five features, and retain the restored honest score. This demonstrates why client holdout alone cannot fix feature leakage.

In [6]:
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(features, y, groups=groups))
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))
assert y.iloc[train_idx].nunique() == 2 and y.iloc[test_idx].nunique() == 2
assert features.iloc[train_idx].notna().any().all()

def quick_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(max_depth=3, min_samples_leaf=1, random_state=42),
    )

def mean_precision_at_20(probabilities):
    scored = pd.DataFrame({
        "client": groups.iloc[test_idx].to_numpy(),
        "content": window_frame.content_hash_id.iloc[test_idx].to_numpy(),
        "label": y.iloc[test_idx].to_numpy(), "score": probabilities,
    })
    sized = scored.groupby("client")["content"].transform("size")
    scored = scored.loc[sized.ge(20)]
    if scored.empty:
        return np.nan
    top = scored.sort_values(["client", "score", "content"],
                             ascending=[True, False, True]).groupby("client").head(20)
    return top.groupby("client")["label"].mean().mean()

honest_model = quick_model()
honest_model.fit(features.iloc[train_idx], y.iloc[train_idx])
honest_probabilities = honest_model.predict_proba(features.iloc[test_idx])[:, 1]
honest_auc = roc_auc_score(y.iloc[test_idx], honest_probabilities)

# This sixth column is deliberately illegal and used only for the lesson.
X_leaky = features.copy()
X_leaky["leak_target_copy"] = y
assert X_leaky.shape[1] == 6
leaky_model = quick_model()
leaky_model.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
leaky_probabilities = leaky_model.predict_proba(X_leaky.iloc[test_idx])[:, 1]
leaky_auc = roc_auc_score(y.iloc[test_idx], leaky_probabilities)

leakage_results = [
    {"run": "Honest: five pre-decision features", "roc_auc": honest_auc,
     "mean_client_precision_at_20": mean_precision_at_20(honest_probabilities)},
    {"run": "Invalid: plus one target-copy column", "roc_auc": leaky_auc,
     "mean_client_precision_at_20": mean_precision_at_20(leaky_probabilities)},
]

# Remove the leak from the inputs and discard the invalid model/predictions.
X_leaky.drop(columns=["leak_target_copy"], inplace=True)
assert list(X_leaky.columns) == FEATURES
del X_leaky, leaky_model, leaky_probabilities

final_feature_frame = features.copy()
assert list(final_feature_frame.columns) == FEATURES and len(FEATURES) == 5
assert "leak_target_copy" not in final_feature_frame.columns
final_model = quick_model()
final_model.fit(final_feature_frame.iloc[train_idx], y.iloc[train_idx])
final_probabilities = final_model.predict_proba(final_feature_frame.iloc[test_idx])[:, 1]
final_auc = roc_auc_score(y.iloc[test_idx], final_probabilities)
assert np.allclose(final_probabilities, honest_probabilities)
assert np.isclose(leaky_auc, 1.0), "The target-copy demonstration should be perfectly separable."

leakage_results.append({
    "run": "Restored: leak removed, five features", "roc_auc": final_auc,
    "mean_client_precision_at_20": mean_precision_at_20(final_probabilities),
})
print(f"Train: {len(train_idx):,} pages, {groups.iloc[train_idx].nunique()} clients; "
      f"test: {len(test_idx):,} pages, {groups.iloc[test_idx].nunique()} clients.")
print(f"Test positive rate: {y.iloc[test_idx].mean():.2%}.")
test_client_sizes = groups.iloc[test_idx].value_counts()
print(f"Precision@20 includes {int(test_client_sizes.ge(20).sum())} test clients; "
      f"{int(test_client_sizes.lt(20).sum())} client(s) have fewer than 20 pages and are excluded.")
print(f"Leaked ROC-AUC gain: {leaky_auc - honest_auc:+.4f}.")
print(f"Final retained honest ROC-AUC: {final_auc:.4f}; final inputs: five features.")
pd.DataFrame(leakage_results).round(4)

Train: 14,070 pages, 24 clients; test: 43,389 pages, 8 clients.
Test positive rate: 38.25%.
Precision@20 includes 7 test clients; 1 client(s) have fewer than 20 pages and are excluded.
Leaked ROC-AUC gain: +0.3876.
Final retained honest ROC-AUC: 0.6124; final inputs: five features.


,run,roc_auc,mean_client_precision_at_20
0,Honest: five pre-decision features,0.6124,0.3571
1,Invalid: plus one target-copy column,1.0000,0.9643
2,"Restored: leak removed, five features",0.6124,0.3571


**What the experiment means:** The invalid result sees the outcome itself, so it is not evidence of predictive skill. The final retained score belongs to the five pre-decision inputs only. A modest honest score is more useful than a perfect leaked result. No feature, threshold, or model was tuned using the test score. The held-out client's later impressions are used only for evaluation.

Precision@20 need not become 1.0 even with the leak: a small client's queue may contain fewer than 20 positive pages. ROC-AUC demonstrates perfect separation without assuming that every queue contains 20 positives.

## 4. Data limits

**Named limitation — one-month complete-case slice:** March alone cannot establish seasonality, sustained decline, or generalization to future months. Requiring complete outcome coverage makes this a retrospective labeled subset; production candidates cannot be filtered by information about their future tracking coverage. Reported scores may therefore be optimistic for sparse-history pages or clients omitted by the coverage/volume rules.

The coverage filter also makes a count of positive-impression days constant in this slice. I checked feature variation and used pre-decision week-to-week impression change as the fifth feature instead. This checks input usability, not which feature gives the best test score; no alternatives were selected by held-out performance.

The group split reserves 25% of clients, not 25% of pages. Client inventories are unequal, so the printed page counts can be very different between train and test. ROC-AUC weights pages, whereas my queue metric weights eligible test clients equally; neither alone establishes generalization beyond this month.

The three-day buffer is an explicit availability assumption because the release lacks historical ingestion timestamps. The same content can lose impressions due to demand changes, consolidation, or other causes; none of these scores proves a page needs a refresh. There is no intervention history or causal design showing that an edit would recover traffic.

Client histories and tracking availability are unbalanced. I use the daily GSC flags and full-window coverage checks here; a larger study must also check client tracking start dates. No GA4-based feature is inferred from zero-filled or unknown GA4 rows. June remains sealed for later final evaluation. All public outputs are aggregate summaries or small derived-feature previews, without real client names, domains, queries, URLs, or credentials.

## 5. Self-check

- [x] Five plain-word contract answers are stated and consistent with the executed outputs.
- [x] Exactly three verification queries ran: grain, row count/date span, and availability using `IS TRUE`.
- [x] Five features were built from March 1–14, with an availability explanation for each.
- [x] The target uses only March 18–31 outcomes and is excluded from the features.
- [x] The deliberate leaked column, score jump, removal, and restored honest score are visible.
- [x] The final feature frame has exactly five columns, with no identifiers or label-derived field.
- [x] Client holdout, train-only imputation, coverage rules, and one-month limitations are explained.
- [x] All cells ran top to bottom without errors and outputs are retained.
- [x] No credentials, raw datasets, or identifying content are included in the commit.

**Submission:** Once all checks above are verified, commit and push the executed notebook at `work/notebooks/w03_data_contract.ipynb`, inspect its outputs on GitHub, and submit the repository URL on this assignment card.